In [1]:
from dataset.preprocess.vballnet import VBallNet
from pathlib import Path

vballnet = VBallNet()

In [2]:

path = Path("/home/skynet/data/sci-volley-vision/VballNet/train/")
manifest_df = vballnet.build_vballnet_manifest(path).split_dataset().preprocess_frames(output_dir="/home/skynet/data/vballnet/processed/512x288-2").get_manifest()

build_vballnet_manifest
Found 15 possible match directories


/home/skynet/research/VolleyVision/dataset/preprocess/vballnet.py:185: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  rng.shuffle(videos)



Dataset summary
----------------
matches: 15
csv_files: 70
valid_videos: 70
missing_videos: 0
invalid_videos: 0
annotations: 23350
split_dataset
preprocess_frames


Processing videos: 100%|██████████| 70/70 [02:27<00:00,  2.10s/it]


In [3]:
from torch.utils.data import DataLoader
from dataset.volley_ball_dataset import VolleyBallDataset

train_df = manifest_df[
    manifest_df["split"] == "train"
]

val_df = manifest_df[
    manifest_df["split"] == "val"
]

dataset = VolleyBallDataset(train_df, sigma=3.0)
loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

val_dataset = VolleyBallDataset(train_df, sigma=3.0)
val_loader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)

In [4]:
batch = next(iter(loader))

print(batch["image"].shape)
print(batch["target"].shape)
print(batch["visibility"].shape)

torch.Size([16, 3, 288, 512])
torch.Size([16, 1, 288, 512])
torch.Size([16])


In [5]:
import torch
from tqdm import tqdm

def train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, device=None):
    if device is None:
        device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
    )

    criterion = torch.nn.MSELoss()

    history = {
        "train_loss": [],
        "val_loss": [],
    }

    for epoch in range(epochs):
        # -------------------------
        # Train
        # -------------------------
        model.train()

        train_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{epochs}",
        )

        for batch in progress:
            images = batch["image"].to(
                device,
                non_blocking=True,
            )

            targets = batch["target"].to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad()

            predictions = model(images)

            loss = criterion(
                predictions,
                targets,
            )

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            progress.set_postfix(
                loss=f"{loss.item():.6f}",
            )

        train_loss /= len(train_loader)

        # -------------------------
        # Validation
        # -------------------------
        model.eval()

        val_loss = 0.0

        with torch.no_grad():
            for batch in val_loader:
                images = batch["image"].to(
                    device,
                    non_blocking=True,
                )

                targets = batch["target"].to(
                    device,
                    non_blocking=True,
                )

                predictions = model(images)

                loss = criterion(
                    predictions,
                    targets,
                )

                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train_loss={train_loss:.6f} | "
            f"val_loss={val_loss:.6f}"
        )

    return model, history

In [ ]:
from model.cnn import CNN

model = CNN()

model = train_model(model=model, train_loader=loader, val_loader=val_loader, epochs=20, lr=1e-3)

Epoch 1/20:  81%|████████  | 807/998 [02:48<00:40,  4.67it/s, loss=0.000192]